# A reproducible PyTorch pipeline for Sonar

Learning outcomes: preserve tensor shapes, pair raw logits with binary cross-entropy,
select and restore a validation checkpoint, compare with a matched baseline,
and verify predictions after saving and reloading.
The 208-row dataset is a teaching example, with no supplied acquisition groups.

Student edition. Reference code is provided; complete the prediction, verification, controlled experiment, and interpretation tasks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "curriculum" / "build.py").exists())
SEED = 42

In [ ]:
import copy
import json
import joblib
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
torch.set_num_threads(1)
folder = ROOT / "neural_networks_course/Day_2_NN_Advanced/exercises/01_pytorch_pipeline"
df = pd.read_csv(folder / "sonar.all-data.csv", header=None)
X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
y = df.iloc[:, -1].map({"R": 0, "M": 1}).to_numpy(dtype=np.float32)
assert X.shape == (208, 60) and np.isfinite(y).all()
DEVICE = torch.device("cpu")  # Small experiment, reproducible on classroom laptops.

In [ ]:
# Three partitions before fitting the scaler.
dev_idx, final_idx = train_test_split(np.arange(len(y)), test_size=0.2,
                                     stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.25,
                                    stratify=y[dev_idx], random_state=SEED)
assert not set(train_idx) & set(val_idx)
assert not set(dev_idx) & set(final_idx)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print("Train / validation / reserved final:", len(train_idx), len(val_idx), len(final_idx))

## 1. Verify preprocessing
Explain why the scaler is fitted only on X_train. Add an assertion showing that the fitted mean equals the raw training mean.

In [ ]:
scaler = StandardScaler().fit(X_train)
Xt = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
Xv = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
yv = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
assert Xt.shape[1] == 60 and yt.shape == (len(Xt), 1)
baseline = LogisticRegression(max_iter=1500).fit(Xt.numpy(), y_train)
baseline_accuracy = accuracy_score(y_val, baseline.predict(Xv.numpy()))
print("Baseline validation accuracy:", baseline_accuracy)

**Your answer:** Explain your reasoning and show the relevant evidence.

## 2. Define the model
Predict its parameter count for width=32. State the output shape and explain why the last layer has no sigmoid.

In [ ]:
def build_model(width=32, dropout=0.2):
    return nn.Sequential(nn.Linear(60, width), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(width, 1)).to(DEVICE)
print("Parameters:", sum(p.numel() for p in build_model().parameters()))

**Your answer:** Explain your reasoning and show the relevant evidence.

## 3. Inspect the training loop
Identify the five operations of each training step, the train/eval transition, and the copied checkpoint. Explain why shallow copying a state dictionary can fail.

In [ ]:
def train_model(width=32, dropout=0.2, lr=0.001, epochs=80, seed=SEED):
    torch.manual_seed(seed)
    model = build_model(width, dropout)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.001)
    history, best_state, best_loss, stale = [], None, float("inf"), 0
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            assert torch.isfinite(loss)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        model.eval()
        with torch.inference_mode():
            val_loss = criterion(model(Xv), yv).item()
        history.append({"epoch": epoch, "train_loss": total / len(Xt), "val_loss": val_loss})
        if val_loss < best_loss - 1e-7:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= 10:
            break
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history), best_epoch

config = dict(width=32, dropout=0.2, lr=0.001, epochs=80, seed=SEED)
model, history, best_epoch = train_model(**config)
print("Selected epoch:", best_epoch)

**Your answer:** Explain your reasoning and show the relevant evidence.

## 4. Diagnose before changing settings

In [ ]:
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(9, 4))
ax.axvline(best_epoch, color="black", linestyle="--", label="Selected epoch")
ax.set_ylabel("Binary cross-entropy")
ax.legend()
plt.show()
with torch.inference_mode():
    val_prob = torch.sigmoid(model(Xv)).numpy().ravel()
print(classification_report(y_val, val_prob >= 0.5,
                            target_names=["Rock", "Mine"], zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_val, val_prob >= 0.5, labels=[0, 1]))

## 5. Controlled experiment
Change one setting in config, state a hypothesis, then rerun training and validation. Keep the split seed, training budget, and decision threshold fixed. Retain the selected config/model together. Compare with the baseline; acknowledge selection optimism and the small validation set.

**Your answer:** State the hypothesis, the one setting you changed, and the evidence, next to the baseline.

## 6. Save and check a complete prediction path
A model file alone does not preserve preprocessing, feature order, labels, or the split.

In [ ]:
run_dir = ROOT / "output/sonar/pytorch"
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), run_dir / "weights.pt")
joblib.dump({"scaler": scaler, "baseline": baseline}, run_dir / "preprocessing.joblib")
history.to_csv(run_dir / "learning_curve.csv", index=False)
metadata = {"config": config, "best_epoch": best_epoch, "features": list(range(60)),
            "labels": {"0": "Rock", "1": "Mine"}, "threshold": 0.5,
            "train_rows": train_idx.tolist(), "validation_rows": val_idx.tolist(),
            "final_rows": final_idx.tolist(), "torch_version": torch.__version__}
(run_dir / "run.json").write_text(json.dumps(metadata, indent=2))
restored = build_model(config["width"], config["dropout"])
restored.load_state_dict(torch.load(run_dir / "weights.pt", weights_only=True, map_location="cpu"))
restored.eval()
saved_scaler = joblib.load(run_dir / "preprocessing.joblib")["scaler"]
check_x = torch.tensor(saved_scaler.transform(X_val), dtype=torch.float32)
with torch.inference_mode():
    check = torch.sigmoid(restored(check_x)).numpy().ravel()
np.testing.assert_allclose(check, val_prob, rtol=1e-6, atol=1e-6)
print("Reload prediction check passed")

## 7. One locked final assessment
Run only after selecting the full procedure. Compare the neural model and baseline on the identical reserved observations. Do not change settings based on this report.

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    final_x = torch.tensor(saved_scaler.transform(X[final_idx]), dtype=torch.float32)
    with torch.inference_mode():
        probability = torch.sigmoid(restored(final_x)).numpy().ravel()
    for name, pred in [("Baseline", baseline.predict(final_x.numpy())),
                       ("Neural", probability >= 0.5)]:
        print(name)
        print(classification_report(y[final_idx], pred,
                                    target_names=["Rock", "Mine"], zero_division=0))

## Submission
Provide learning curves, the selected checkpoint and config, reload evidence, baseline comparison, and one error analysis. Restart the kernel and run in order before submission. Explain what this small, ungrouped dataset cannot establish about new acquisition conditions.